# 04. Abliacija ir klaidų analizė

> **Peržiūros notebook.** Šaltiniai: `reports/tables/ablation.csv`, `reports/error_cases.csv`, `reports/error_profile.txt`.


## Kas čia vyksta paprastai

Abliacija klausia: *ar RBF branduolys tikrai reikalingas? ar užtenka 10 „mean“ ar 10 „worst“ požymių? ar kaina 5:1 keičia slenkstį?* Tai daroma su **tais pačiais** užšaldytais modeliais ir testu — ne nauju HP tinklu ant testo.

Klaidų analizė žiūri **konkrečius** 2 FN ir 5 FP SVM-RBF ties `t_se98=0,34`: ar jie „ant slenksčio“, ar morfologiškai panašūs į priešingą klasę.


## `evalx/ablation.m` — lentelė `ablation.csv`

| id | setting | t | Se | Sp | FN | FP | AUC | BS | EC arba Δ | pastaba |
|---|---|---:|---:|---:|---:|---:|---:|---:|---:|---|
| A0 | majority_t05 | 0,50 | 0 | 1 | 42 | 0 | 0,500 | 0,372 | — | B0 kartelė |
| A2 | svm_rbf_tse98 | 0,34 | 0,9524 | 0,9296 | 2 | 5 | 0,9863 | 0,0430 | ΔSp vs lin. +0,042 | dSp PI [−0,029; 0,111] |
| A2 | svm_linear_tse98 | 0,16 | 0,9762 | 0,8873 | 1 | 8 | **0,9953** | 0,0312 | ΔSp vs RBF −0,042 | dSp PI [−0,111; 0,029] |
| A5 | mean10_tse98 | 0,08 | 0,9762 | 0,7183 | 1 | 20 | 0,9745 | 0,0619 | — | Sp PI [0,615; 0,819] |
| A5 | worst10_tse98 | 0,09 | 1,000 | 0,8732 | 0 | 9 | **0,9963** | 0,0249 | — | Sp PI [0,800; 0,945] |
| A5 | all30_tse98 | 0,34 | 0,9524 | 0,9296 | 2 | 5 | 0,9863 | 0,0430 | — | = M1, Sp PI [0,864; 0,986] |
| A7_1_1 | svm_t_cost | 0,33 | 0,9524 | 0,9296 | 2 | 5 | 0,9863 | 0,0430 | EC=7 | |
| A7_3_1 | svm_t_cost | 0,33 | 0,9524 | 0,9296 | 2 | 5 | 0,9863 | 0,0430 | EC=11 | |
| A7_5_1 | svm_t_cost | 0,33 | 0,9524 | 0,9296 | 2 | 5 | 0,9863 | 0,0430 | EC=15 | |
| A7_10_1 | svm_t_cost | 0,33 | 0,9524 | 0,9296 | 2 | 5 | 0,9863 | 0,0430 | EC=25 | |

**A0.** Dauguma: visus 42 M praleidžia. Be šios eilutės AUC ≈ 0,99 atrodytų „savaime“.

**A2.** Tiesinis SVM **didesnis** AUC nei RBF (0,995 vs 0,986). RBF ΔSp vs linear taškas +0,042, bet 95 % PI **kerta 0**. RBF branduolys šioje imtyje **nėra** įrodytas reikalingas rangui; Sp skirtumas nepatikimas.

**A5.** `worst` 10 požymių: AUC 0,996, Se=1, Sp=0,873. `mean` 10: Sp tik 0,718, 20 FP. Visi 30 geriausias Sp tarp A5, bet ne AUC. „Worst“ statistikos neša daug atskyrimo; „mean“ vienos nepakanka Sp.

**A7.** Tas pats `t=0,33`, FN=2, FP=5 visiems 1:1, 3:1, 5:1, 10:1. Keičiasi tik **skaliarinis** EC. Kainų santykis šioje imtyje slenksčio **nepastūmė**.


## Grafikai, susiję su abliacija ir kaina

![ROC — visos kreivės, įskaitant linear / mean / worst](../reports/figures/roc.png)

**Parašas.** Tos pačios testo ROC kaip 03: `svm_linear` ir `svm_worst` plotu neatsilieka nuo RBF (atitinka A2/A5 AUC stulpelius).

![EC kreivė](../reports/figures/cost_curve.png)

**Parašas.** A7: EC proporcingas svoriams, bet arg min t ir FN/FP nesikeičia (`ablation.csv` A7_* eilutės).


## `evalx/error_analysis.m`

**Paprastai:** imami SVM-RBF klaidingi sprendimai teste ir palyginami su mokymo medianomis M ir B pagal `concave_points_worst` ir `area_worst`.

```matlab
% evalx/error_analysis.m (fragmentas)
t = pack.svm_rbf.thresholds.t_se98;   % 0.34
yhat = p >= t;
fn = find(ytest == 1 & ~yhat);
fp = find(ytest == 0 & yhat);
% dist_to_t = |p - t|; cp/ar minus mokymo medianos M ir B
```

`error_profile.txt`:

```
t_se98=0.3400  FN=2  FP=5
train median M: cp_worst=0.1827 area_worst=1332.5 texture_mean=21.52 smoothness_worst=0.1419
train median B: cp_worst=0.07412 area_worst=548.45 texture_mean=17.31 smoothness_worst=0.12345
FN median: cp_worst=0.09844 area_worst=888.25
FP median: cp_worst=0.1571 area_worst=614.9
```

FN medianos **tarp** M ir B (plotas didesnis nei tipinis B, įgaubos taškai arčiau B). FP medianos: įgaubos arčiau M, plotas vis dar B zonoje — „mažas, bet nelygus kontūras“.


## FN / FP atvejai — `error_cases.csv`

SVM-RBF, `t=0,34`. `abs_idx` — eilutė visame 569 rinkinyje (1-indeksuota MATLAB).

| tipas | abs_idx | p | \|p−t\| | interpretacija |
|---|---:|---:|---:|---|
| FN | 41 | 0,048 | 0,292 | toli žemiau slenksčio; ne „riboje“ |
| FN | 264 | 0,214 | 0,126 | vis dar aiškiai < 0,34 |
| FP | 69 | 0,776 | 0,436 | toli virš t; mažas `area`, dideli concave_points |
| FP | 82 | 0,508 | 0,168 | |
| FP | 153 | 0,804 | 0,464 | panašiai kaip 69 |
| FP | 291 | 0,446 | 0,106 | |
| FP | 466 | 0,385 | **0,045** | vienintelis aiškiai **ribinis** |

Abu FN **nėra** slenksčio artefaktas: FN 41 p=0,048 liktų B net prie LR `t_se98=0,21`. FP 466 — kandidatas `margin_flag` prototipe (`|p−t|<0,10`).


## Painiavos matrica (tie patys 2 FN, 5 FP)

![SVM-RBF confusion](../reports/figures/confusion_svm_rbf.png)

**Parašas.** Vizualus `error_cases.csv` suskaičiavimas: 2 + 5 klaidos iš 113.

Toliau: [05_prototipas_ir_ai_zurnalas.ipynb](05_prototipas_ir_ai_zurnalas.ipynb).
